# Runtime Context

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "green"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-sonnet-4-5",
    context_schema=ColourContext
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='8822f7de-2aad-407a-b538-f6664fad65e4'),
              AIMessage(content="I don't have any information about your favorite colour. We haven't spoken before, and you haven't told me what it is yet. \n\nWould you like to tell me what your favorite colour is?", additional_kwargs={}, response_metadata={'id': 'msg_015xNUexfwVbYnpESX3RHu9K', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 13, 'output_tokens': 45, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbd2-7e90-74a1-9431-7f1a5dd470a1-0', to

# Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model="claude-sonnet-4-5",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

/Users/saketmunda/Work/Learn/langchain-atoz/lca-lc-foundations/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='green'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='2ba28072-3efc-4df2-8f6b-299aa8c454f1'),
              AIMessage(content=[{'id': 'toolu_01Aj3zBNZRyhgsHWNBCo31QU', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01WqXF8Bu7PZt1CQt3FaK4xX', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 598, 'output_tokens': 38, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbd2-886b-7631-9766-8d4b97953842-0', tool_calls=[{'name': 'get_favourite_colour', 'ar

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="red")
)

pprint(response)

/Users/saketmunda/Work/Learn/langchain-atoz/lca-lc-foundations/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='green'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='d9817a38-b463-482e-b79b-347db49e32ae'),
              AIMessage(content=[{'id': 'toolu_01RSyx5hth6w6ki1u6FsrVqo', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01TUNEwfF6qoV5dcwgzFK5Y3', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 598, 'output_tokens': 38, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbd6-5c88-7450-881e-88824eb8b9ad-0', tool_calls=[{'name': 'get_favourite_colour', 'ar

# State

In [10]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to State

In [11]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages":[ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "claude-sonnet-4-5",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [13]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is black")]},
    config=config
)

In [14]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'black',
 'messages': [HumanMessage(content='My favourite colour is black', additional_kwargs={}, response_metadata={}, id='5c4a0c7c-608e-4eb6-a30e-44f20138481c'),
              AIMessage(content=[{'id': 'toolu_013vEcYWBQB8RPRq1xbXdqjQ', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'black'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_015HcEqsHsEcq94Wkv4aWKhr', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 577, 'output_tokens': 57, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbe8-7c94-7053-aab8-b4672

## Read State

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


agent = create_agent(
    model="claude-sonnet-4-5",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [16]:
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is black")]},
    config=config
)

pprint(response)

{'favourite_colour': 'black',
 'messages': [HumanMessage(content='My favourite colour is black', additional_kwargs={}, response_metadata={}, id='f36bb504-d80e-4627-a23c-03e6507ffdc8'),
              AIMessage(content=[{'id': 'toolu_0183Jcmpy7CTGakDSZVk8GxH', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'black'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01RktSNVbtdVFiBvBLA3vMEu', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 623, 'output_tokens': 57, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbff-4b7f-7523-8c83-21ff2

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    config=config
)

pprint(response)

{'favourite_colour': 'black',
 'messages': [HumanMessage(content='My favourite colour is black', additional_kwargs={}, response_metadata={}, id='f36bb504-d80e-4627-a23c-03e6507ffdc8'),
              AIMessage(content=[{'id': 'toolu_0183Jcmpy7CTGakDSZVk8GxH', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'black'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01RktSNVbtdVFiBvBLA3vMEu', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 623, 'output_tokens': 57, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019dfbff-4b7f-7523-8c83-21ff2